In [2]:
import pandas as pd

### what columns do we have

In [4]:
data = pd.read_csv('../../data/task_2_data_ex.csv')
data.head()

,year,month,produced_material,produced_material_production_type,produced_material_release_type,produced_material_quantity,component_material,component_material_production_type,component_material_release_type,component_material_quantity,plant_id
0,2024,1,10000,8002,FIN,990.00,50000,8002.0,PROD,990.00,RLT_10
1,2024,1,50000,8002,PROD,859.00,80070,8007.0,PROD,879.00,RLT_10
2,2024,1,50000,8002,PROD,859.00,90000,NaN,ADD,50.00,RLT_10
3,2024,1,50000,8002,PROD,859.00,90001,NaN,ADD,20.00,RLT_10
4,2024,1,80070,8007,PROD,929.00,80010,8001.0,PROD,"3,626.00",RLT_10


### analyse the numeric values in the dataset
we can see that we have 2024 year with up to 12 month (not as 2025 year and 0 month)

In [13]:
data.describe()

,year,month,produced_material,produced_material_production_type,component_material,component_material_production_type
count,1320.0,1320.000000,1320.000000,1320.000000,1320.000000,480.000000
mean,2024.0,6.500000,65479.954545,8002.818182,81841.136364,8002.500000
std,0.0,3.453361,21916.019385,2.657669,11933.718591,2.695392
min,2024.0,1.000000,10000.000000,8000.000000,50000.000000,8000.000000
25%,2024.0,3.750000,50005.000000,8001.000000,80007.000000,8000.750000
50%,2024.0,6.500000,80007.000000,8002.000000,90004.500000,8001.500000
75%,2024.0,9.250000,80070.000000,8007.000000,90027.000000,8003.250000
max,2024.0,12.000000,80079.000000,8007.000000,90050.000000,8007.000000


### we have a lot of NaN data in ```component_material_production_type``` feature

because of Raw Materials (RM) and Additives (ADD). they don't have any components (they are leaves in the hierarchy tree)

In [14]:
data.isnull().sum()

year                                    0
month                                   0
produced_material                       0
produced_material_production_type       0
produced_material_release_type          0
produced_material_quantity              0
component_material                      0
component_material_production_type    840
component_material_release_type         0
component_material_quantity             0
plant_id                                0
dtype: int64

### we can see that `produced_material_quantity` and `component_material_quantity` has `str` type (`not int`)

In [15]:
data.dtypes

year                                    int64
month                                   int64
produced_material                       int64
produced_material_production_type       int64
produced_material_release_type            str
produced_material_quantity                str
component_material                      int64
component_material_production_type    float64
component_material_release_type           str
component_material_quantity               str
plant_id                                  str
dtype: object

# Business logic
1. extract data from csv file and process float nums into valid values
2. concatinate all necessary time to year to create FIN (aggregate from monthly to annualy)
3. building indexing (like B+ tree) to speed up the search
4. for each FIN and PROD create a new hierarchy raw
5. final report of created new DataFrame

In [5]:
EXPLODEABLE = {"FIN", "PROD"}  # ADD and RM are leaves

In [6]:
def load_and_clean_bom(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    qty_cols = ["produced_material_quantity", "component_material_quantity"]
    for col in qty_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .astype(float)
        )

    return df

In [7]:
def aggregate_bom_to_year(df: pd.DataFrame) -> pd.DataFrame:
    """One BoM edge per plant/year/material/component, quantities summed."""
    group_cols = [
        "plant_id", "year",
        "produced_material", "produced_material_production_type", "produced_material_release_type",
        "component_material", "component_material_production_type", "component_material_release_type",
    ]

    return (
        df.groupby(group_cols, dropna=False, as_index=False)
        .agg(
            produced_material_quantity=("produced_material_quantity", "sum"),
            component_material_quantity=("component_material_quantity", "sum"),
        )
    )

In [8]:
def build_bom_index(df: pd.DataFrame) -> dict[tuple, pd.DataFrame]:
    """(plant, year, produced_material) -> rows of its components."""
    return {
        key: group.reset_index(drop=True)
        for key, group in df.groupby(["plant_id", "year", "produced_material"], sort=False)
    }

In [9]:
def explode_fin_material(
    bom_index: dict,
    plant: str,
    year: int,
    fin_material: int,
    fin_meta: pd.Series,
) -> list[dict]:
    rows = []
    queue = [fin_material]
    seen = set()

    while queue:
        material = queue.pop(0)
        key = (plant, year, material)
        if key in seen:
            continue
        seen.add(key)

        for _, edge in bom_index.get(key, pd.DataFrame()).iterrows():
            rows.append({
                "plant": plant,
                "year": year,
                "fin_material_id": fin_material,
                "fin_material_release_type": fin_meta["produced_material_release_type"],
                "fin_material_production_type": fin_meta["produced_material_production_type"],
                "fin_production_quantity": fin_meta["produced_material_quantity"],
                "prod_material_id": edge["produced_material"],
                "prod_material_release_type": edge["produced_material_release_type"],
                "prod_material_production_type": edge["produced_material_production_type"],
                "prod_material_production_quantity": edge["produced_material_quantity"],
                "component_id": edge["component_material"],
                "component_material_release_type": edge["component_material_release_type"],
                "component_material_production_type": edge["component_material_production_type"],
                "component_consumption_quantity": edge["component_material_quantity"],
            })

            if edge["component_material_release_type"] in EXPLODEABLE:
                queue.append(edge["component_material"])

    return rows

In [10]:
def explode_all(bom_annual: pd.DataFrame) -> pd.DataFrame:
    bom_index = build_bom_index(bom_annual)

    fin_roots = (
        bom_annual[bom_annual["produced_material_release_type"] == "FIN"]
        .drop_duplicates(["plant_id", "year", "produced_material"])
    )

    all_rows = []
    for _, fin in fin_roots.iterrows():
        all_rows.extend(
            explode_fin_material(
                bom_index,
                fin["plant_id"],
                fin["year"],
                fin["produced_material"],
                fin,
            )
        )

    return pd.DataFrame(all_rows)

In [11]:
def run_pipeline(path: str) -> pd.DataFrame:
    data = load_and_clean_bom(path)
    bom_annual = aggregate_bom_to_year(data)
    exploded = explode_all(bom_annual)
    return exploded

In [12]:
run_pipeline(path="../../data/task_2_data_ex.csv")

,plant,year,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity
0,RLT_10,2024,10000,FIN,8002,11708.0,10000,FIN,8002,11708.0,50000,PROD,8002.0,11708.0
1,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,80070,PROD,8007.0,11303.0
2,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90000,ADD,NaN,598.0
3,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90001,ADD,NaN,242.0
4,RLT_10,2024,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,80010,PROD,8001.0,41769.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,RLT_16,2024,10008,FIN,8002,12067.0,80078,PROD,8007,10676.0,90043,ADD,NaN,120.0
106,RLT_16,2024,10008,FIN,8002,12067.0,80018,PROD,8001,21410.0,80008,PROD,8000.0,23516.0
107,RLT_16,2024,10008,FIN,8002,12067.0,80018,PROD,8001,21410.0,90044,ADD,NaN,1198.0
108,RLT_16,2024,10008,FIN,8002,12067.0,80008,PROD,8000,23397.0,70008,RM,NaN,31048.0
